In [7]:
import sys
import pandas as pd
import numpy as np

print("¡Entorno de Colab conectado con éxito para ARQIA!")
print(f"Versión de Python: {sys.version.split()[0]}")
print(f"Pandas instalado: {pd.__version__}")
print(f"NumPy instalado: {np.__version__}")


¡Entorno de Colab conectado con éxito para ARQIA!
Versión de Python: 3.13.18
Pandas instalado: 2.2.3
NumPy instalado: 2.4.6


In [8]:
import pandas as pd

# Cargar dataset sintético desde data/
df_audit = pd.read_csv("data/triaje_simulado_arqia.csv")

# 1. ¿Cuántos datos y qué estructura tenemos?
print(f"Registros (filas): {df_audit.shape[0]} | Columnas: {df_audit.shape[1]}")
print("\nTipos de datos:")
print(df_audit.dtypes)

# 2. Auditoría de calidad y reglas de limpieza
nulos = df_audit.isnull().sum()
duplicados = df_audit["id_registro"].duplicated().sum()
cols_escala = [
    "carga_laboral",
    "autonomia",
    "soporte_social",
    "recuperacion_descanso",
]
fuera_rango = ((df_audit[cols_escala] < 1) | (df_audit[cols_escala] > 10)).sum()

print("\nNulos por columna:\n", nulos)
print(f"\nIDs duplicados: {duplicados}")
print("\nValores fuera del rango [1, 10]:\n", fuera_rango)

# 3. Limpieza preventiva y enlace directo hacia el cálculo del ISP
df_limpio = df_audit.drop_duplicates(subset=["id_registro"]).copy()
for col in cols_escala:
  df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce").clip(1, 10)
df_limpio = df_limpio.dropna(subset=cols_escala).reset_index(drop=True)

# Dejamos df_sim listo para la celda siguiente del ISP
df_sim = df_limpio.copy()

print(f"\nRegistros listos para el baseline ISP: {len(df_sim)}")
print("\nResumen descriptivo (1 a 10):")
print(df_sim[cols_escala].describe().round(2))

Registros (filas): 50 | Columnas: 6

Tipos de datos:
id_registro               object
carga_laboral              int64
autonomia                  int64
soporte_social             int64
recuperacion_descanso      int64
ISP_Score                float64
dtype: object

Nulos por columna:
 id_registro              0
carga_laboral            0
autonomia                0
soporte_social           0
recuperacion_descanso    0
ISP_Score                0
dtype: int64

IDs duplicados: 0

Valores fuera del rango [1, 10]:
 carga_laboral            0
autonomia                0
soporte_social           0
recuperacion_descanso    0
dtype: int64

Registros listos para el baseline ISP: 50

Resumen descriptivo (1 a 10):
       carga_laboral  autonomia  soporte_social  recuperacion_descanso
count          50.00      50.00           50.00                  50.00
mean            5.20       5.44            5.26                   4.38
std             2.49       2.80            2.65                   2.54
min   

In [9]:
import pandas as pd
import numpy as np

# Configuración de semilla y datos sintéticos de triaje
np.random.seed(42)
n_registros = 50

df_triaje = pd.DataFrame({
    "id_registro": [f"REG_{i:03d}" for i in range(1, n_registros + 1)],
    "carga_laboral": np.random.randint(1, 10, size=n_registros),
    "autonomia": np.random.randint(1, 10, size=n_registros),
    "soporte_social": np.random.randint(1, 10, size=n_registros),
    "recuperacion_descanso": np.random.randint(1, 10, size=n_registros)
})

# Cálculo ponderado del ISP (Índice de Sostenibilidad Psicofísica)
def calcular_isp(row):
    estres = row['carga_laboral'] * 0.4
    recursos = (row['autonomia'] * 0.2 + row['soporte_social'] * 0.2 + row['recuperacion_descanso'] * 0.2)
    score = max(0, min(100, 50 - (estres * 5) + (recursos * 10)))
    return round(score, 2)

df_triaje['ISP_Score'] = df_triaje.apply(calcular_isp, axis=1)

# Visualizar los primeros resultados del triaje
print(df_triaje[['id_registro', 'carga_laboral', 'ISP_Score']].head(10))


  id_registro  carga_laboral  ISP_Score
0     REG_001              7       64.0
1     REG_002              4       60.0
2     REG_003              8       58.0
3     REG_004              5       56.0
4     REG_005              7       64.0
5     REG_006              3       72.0
6     REG_007              7       70.0
7     REG_008              8       58.0
8     REG_009              5       56.0
9     REG_010              4       76.0


In [10]:
# Guardar el resultado procesado en la carpeta data/
df_triaje.to_csv('data/triaje_simulado_arqia.csv', index=False)
print("¡Dataset de triaje simulado guardado exitosamente en data/!")


¡Dataset de triaje simulado guardado exitosamente en data/!


In [11]:
import os

os.makedirs("data", exist_ok=True)
df_triaje.to_csv("data/triaje_simulado_arqia.csv", index=False)
print("Guardado OK en data/triaje_simulado_arqia.csv")

Guardado OK en data/triaje_simulado_arqia.csv


In [12]:
import numpy as np
import pandas as pd

# 1. Cargar el dataset sintético versionado en data/
df_sim = pd.read_csv("data/triaje_simulado_arqia.csv")

# 2. Calcular un Índice de Sostenibilidad Psicofísica (ISP) base (escala 0 a 100)
# Factores protectores: autonomía, soporte_social, recuperacion_descanso
# Factor de exigencia/presión: carga_laboral
isp_bruto = (
    0.30 * df_sim["recuperacion_descanso"]
    + 0.25 * df_sim["autonomia"]
    + 0.25 * df_sim["soporte_social"]
    - 0.30 * df_sim["carga_laboral"]
)

# Normalizamos a un rango de 0 a 100 para facilitar la lectura de triaje
df_sim["isp_score"] = np.clip(((isp_bruto + 2.7) / 11.4) * 100, 0, 100).round(
    1
)


# 3. Clasificar en niveles operativos de triaje
def clasificar_triaje(score):
  if score < 35:
    return "Alerta Crítica (Prioridad SST)"
  elif score < 60:
    return "Monitoreo Preventivo"
  else:
    return "Sostenibilidad Estable"


df_sim["nivel_triaje"] = df_sim["isp_score"].apply(clasificar_triaje)

# 4. Resumen de resultados ("dato mata relato")
print("Distribución por nivel de triaje:")
print(df_sim["nivel_triaje"].value_counts())

print("\nPerfiles en Alerta Crítica:")
criticos = df_sim[df_sim["nivel_triaje"] == "Alerta Crítica (Prioridad SST)"]
print(
    criticos[[
        "id_registro",
        "carga_laboral",
        "autonomia",
        "soporte_social",
        "recuperacion_descanso",
        "isp_score",
    ]].head(10)
)

Distribución por nivel de triaje:
nivel_triaje
Monitoreo Preventivo              28
Alerta Crítica (Prioridad SST)    14
Sostenibilidad Estable             8
Name: count, dtype: int64

Perfiles en Alerta Crítica:
   id_registro  carga_laboral  autonomia  soporte_social  \
1      REG_002              4          2               5   
2      REG_003              8          6               3   
3      REG_004              5          6               1   
7      REG_008              8          2               9   
8      REG_009              5          4               3   
10     REG_011              8          7               1   
14     REG_015              5          2               7   
24     REG_025              3          1               4   
28     REG_029              3          1               1   
32     REG_033              5          1               4   

    recuperacion_descanso  isp_score  
1                       2       33.8  
2                       3       30.3  
3        

In [13]:
# Guardar el DataFrame procesado en la carpeta data/
df_triaje.to_csv('data/triaje_simulado_arqia.csv', index=False)
print("¡Dataset de triaje simulado guardado exitosamente en data/!")


¡Dataset de triaje simulado guardado exitosamente en data/!
